In [1]:
import sys
print(sys.executable)

/Users/cindychou/Desktop/4120_nlp/NLP-Final-Project/venv/bin/python


In [2]:
# imports
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.text import Tokenizer
import numpy as np
from keras.preprocessing.sequence import pad_sequences

In [3]:
# view the three datasets
import pandas as pd
import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve()))

TRANSCRIPTS_PATH = Path("..") / "data" / "transcripts_cleaned.csv"

In [4]:
transcript_df = pd.read_csv(TRANSCRIPTS_PATH)
print(transcript_df.shape)
transcript_df.head()

(157, 7)


,Record-ID,Class,Transcript_PFT,Transcript_CTD,Transcript_SFT,Label,Binary_Label
0,Process-rec-001,MCI,"people, partner, plate, platter, pants, porter...",NaN,"<pause_medium> giraffe, kangaroo, lion, tiger,...",1,1
1,Process-rec-002,MCI,"<pause_short> pipe, plane, people <pause_mediu...",<pause_medium> there’s a lad stood on the stoo...,"<pause_short> dogs, cats, birds <pause_short> ...",1,1
2,Process-rec-003,MCI,"um <pause_short> purple, pale, placid <pause_s...","<pause_medium> um, the picture is of a kitchen...","cow, bull, ewe, ram, chicken, goose, um <sigh>...",1,1
3,Process-rec-004,MCI,plank <pause_short> pool <pause_short> swimmin...,"a mother presumably, or a fe, an adult female ...",um <pause_short> impala <pause_short> er cheet...,1,1
4,Process-rec-005,MCI,"it’s er pillock, er post box, er pyracanthas, ...","‘50s style er scene of domestic um confusion, ...","dog, cat, giraffe, wallaby, kangaroo, tortoise...",1,1


In [5]:
TRANSCRIPT_COLS = ["Transcript_PFT", "Transcript_CTD", "Transcript_SFT"]

print("NaN counts per transcript type: ")
for col in TRANSCRIPT_COLS:
    n_nans = transcript_df[col].isna().sum()
    print(f"{col}: {n_nans} NaNs")

NaN counts per transcript type: 
Transcript_PFT: 0 NaNs
Transcript_CTD: 1 NaNs
Transcript_SFT: 5 NaNs


In [6]:
transcript_df.columns

Index(['Record-ID', 'Class', 'Transcript_PFT', 'Transcript_CTD',
       'Transcript_SFT', 'Label', 'Binary_Label'],
      dtype='object')

In [26]:
# pft_df = transcript_df[["Transcript_PFT", "Label"]]
# ctd_df = transcript_df[['Transcript_CTD', 'Label']]
sft_df = transcript_df[['Transcript_SFT', 'Binary_Label']]

In [8]:
print("Phonemic Fluency Test \n", pft_df.head)

Phonemic Fluency Test 
 <bound method NDFrame.head of                                         Transcript_PFT  Label
0    people, partner, plate, platter, pants, porter...      1
1    <pause_short> pipe, plane, people <pause_mediu...      1
2    um <pause_short> purple, pale, placid <pause_s...      1
3    plank <pause_short> pool <pause_short> swimmin...      1
4    it’s er pillock, er post box, er pyracanthas, ...      1
..                                                 ...    ...
152  um, precise, prescient, er procrastination, pr...      0
153  picture, plate, palm, photo, psychology, um <p...      0
154  countries beginning with p: paraguay, portugal...      0
155  phew, phew, phew <pause_long> phidi, philadelp...      2
156  <pause_medium> er <pause_short> paternity, pet...      0

[157 rows x 2 columns]>


In [63]:
print("Category Test: Delayed Recall \n", ctd_series.head)

Category Test: Delayed Recall 
 <bound method NDFrame.head of 0                                                    NaN
1      <pause_medium> there’s a lad stood on the stoo...
2      <pause_medium> um, the picture is of a kitchen...
3      a mother presumably, or a fe, an adult female ...
4      ‘50s style er scene of domestic um confusion, ...
                             ...                        
152    the sink is overflowing; the woman doing the w...
153    i see a scene of absolute chaos in this pictur...
154    little boy falling off a chair whilst passing ...
155    <pause_medium> er, little boy stood on a stool...
156    <pause_medium> ok, well there’s a boy stood on...
Name: Transcript_CTD, Length: 152, dtype: object>


In [27]:
print("Semantic Fluency Test \n", sft_df.head)

Semantic Fluency Test 
 <bound method NDFrame.head of                                         Transcript_SFT  Binary_Label
0    <pause_medium> giraffe, kangaroo, lion, tiger,...             1
1    <pause_short> dogs, cats, birds <pause_short> ...             1
2    cow, bull, ewe, ram, chicken, goose, um <sigh>...             1
3    um <pause_short> impala <pause_short> er cheet...             1
4    dog, cat, giraffe, wallaby, kangaroo, tortoise...             1
..                                                 ...           ...
152  um, well er cat, dog, rabbit, hamster, guinea ...             0
153  cat, lion, tiger oh. cat, lion, tiger, elephan...             0
154  horse, dog, cat, pig, hen <pause_short> walrus...             0
155  pig, cat, dog pig, cat, dog <pause_short> gira...             1
156  ooh. armadillo, antelope, bear, buffalo, er <p...             0

[157 rows x 2 columns]>


# Summary of best configs

In [28]:
from typing import Tuple, Optional, List

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.base import clone
from sklearn.model_selection import GridSearchCV

from transcript_preprocessing import get_stratified_kfold_splits


def train_eval_log_reg_text_only_kfold(
    df: pd.DataFrame,
    text_col: str,
    label_col: str,
    n_splits: int = 5,
    random_state: int = 42,
) -> Tuple[Pipeline, np.ndarray, float, float, dict]:
    """
    Text-only logistic regression, using ONLY Meggan's get_stratified_kfold_splits
    for splitting.

    Steps:
      1. Drop NaNs, build X (text) and y (labels).
      2. Build TF-IDF + LogisticRegression pipeline.
      3. Use Meggan's helper to get stratified folds.
      4. Run GridSearchCV over those same folds to find best hyperparameters.
      5. Using the best hyperparameters, run K-fold evaluation:
         - fit on each fold's train indices
         - evaluate on that fold's test indices
         - print classification report + confusion matrix
         - record macro F1
      6. Print mean/std macro F1 across folds.
      7. Retrain final model on ALL data with best hyperparameters.
      8. Return (final_model, fold_scores, mean_f1, std_f1, best_params).
    """

    # 1. Drop rows with missing text or label
    df = df.dropna(subset=[text_col, label_col]).copy().reset_index(drop=True)

    # 2. X and y (keep X as a DataFrame so ColumnTransformer can see column names)
    X = df[[text_col]]
    y = df[label_col]

    # 3. Define preprocessing: TF-IDF on the text column
    text_transformer = TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_features=10000,
    )

    preprocess = ColumnTransformer(
        transformers=[
            ("text", text_transformer, text_col),
        ],
        remainder="drop",
    )

    # 4. Base Logistic Regression model
    base_log_reg = LogisticRegression(
        solver="lbfgs", 
        max_iter=1000,
        class_weight="balanced",
        n_jobs=-1,
    )

    base_clf = Pipeline(steps=[
        ("preprocess", preprocess),
        ("logreg", base_log_reg),
    ])

    # 5. Use Meggan's helper to get stratified folds (once)
    splits = list(
        get_stratified_kfold_splits(
            transcript_df=df,
            transcript_col=text_col,
            label_col=label_col,
            n_splits=n_splits,
            seed=random_state,
        )
    )

    # For GridSearchCV, we need (train_idx, test_idx) pairs without fold_idx
    cv_for_grid = [(train_idx, test_idx) for (_, train_idx, test_idx) in splits]

    # 6. Grid search over C (and any other hyperparameters you want)
    param_grid = {
        "logreg__C": [0.01, 0.1, 1, 10, 100, 1000, 10000, 100000],
        "logreg__solver": ["lbfgs", "newton-cg"],  # can add others if you want
        "preprocess__text__ngram_range": [(1, 1), (1, 2), (2, 2)],
        "preprocess__text__min_df": [5, 6, 7, 8 , 9, 10],
        "preprocess__text__max_df": [0.8, 0.9, 1.0],
        "preprocess__text__max_features": [50, 60, 70, 80, 90, 100, 200, 300, 400, 500, 1000],
    }

    grid = GridSearchCV(
        estimator=base_clf,
        param_grid=param_grid,
        cv=cv_for_grid,         # uses the SAME splits as Meggan's helper
        scoring="f1_macro",
        n_jobs=-1,
    )

    grid.fit(X, y)
    best_clf_template: Pipeline = grid.best_estimator_
    best_params: dict = grid.best_params_

    print("\n=== Global GridSearchCV (using Meggan's splits) ===")
    print("Best params:", best_params)
    print(f"Best CV f1_macro: {grid.best_score_:.4f}")

    # 7. Now do explicit K-fold evaluation with the best hyperparameters
    fold_scores: List[float] = []

    for fold_idx, train_idx, test_idx in splits:
        print(f"\n=== Fold {fold_idx} / {n_splits} ===")

        X_train = X.iloc[train_idx]
        y_train = y.iloc[train_idx]
        X_test = X.iloc[test_idx]
        y_test = y.iloc[test_idx]

        # Clone the best pipeline so each fold starts fresh, BUT with best_params
        clf_fold = clone(best_clf_template)
        clf_fold.fit(X_train, y_train)

        y_pred = clf_fold.predict(X_test)

        # Classification report for this fold
        print(classification_report(y_test, y_pred))

        # Confusion matrix for this fold
        cm = confusion_matrix(y_test, y_pred)
        print("Confusion matrix (rows = true, cols = predicted):")
        print(cm)

        # Macro F1 for this fold
        f1 = f1_score(y_test, y_pred, average="macro")
        fold_scores.append(f1)
        print(f"Fold {fold_idx} macro F1: {f1:.4f}")

    fold_scores = np.array(fold_scores)
    mean_f1 = fold_scores.mean()
    std_f1 = fold_scores.std()

    print(f"\n=== Overall ({n_splits}-fold, using Meggan's splits) ===")
    print(f"Macro F1: {mean_f1:.4f} ± {std_f1:.4f}")

    # 8. Retrain final model on ALL data with the best hyperparameters
    final_clf = clone(best_clf_template)
    final_clf.fit(X, y)

    return final_clf, fold_scores, mean_f1, std_f1, best_params


In [29]:
results_by_transcript = {}

for df, text_col, name in [
    #(pft_df, "Transcript_PFT", "PFT"),
    #(ctd_df, "Transcript_CTD", "CTD"),
    (sft_df, "Transcript_SFT", "SFT"),
]:
    print(f"\n##### {name} ({text_col}) #####")
    final_clf, fold_scores, mean_f1, std_f1, best_params = train_eval_log_reg_text_only_kfold(
        df=df,
        text_col=text_col,
        label_col="Binary_Label",
        n_splits=5,
        random_state=42,
    )

    results_by_transcript[name] = {
        "final_clf": final_clf,
        "fold_scores": fold_scores,
        "mean_f1": mean_f1,
        "std_f1": std_f1,
        "best_params": best_params,
    }

print("\n=== Summary of best configs ===")
for name, info in results_by_transcript.items():
    print(f"{name}:")
    print("  best_params:", info["best_params"])
    print(f"  mean macro F1: {info['mean_f1']:.4f} ± {info['std_f1']:.4f}")



##### SFT (Transcript_SFT) #####

=== Global GridSearchCV (using Meggan's splits) ===
Best params: {'logreg__C': 100000, 'logreg__solver': 'lbfgs', 'preprocess__text__max_df': 1.0, 'preprocess__text__max_features': 80, 'preprocess__text__min_df': 8, 'preprocess__text__ngram_range': (1, 2)}
Best CV f1_macro: 0.6790

=== Fold 1 / 5 ===
              precision    recall  f1-score   support

           0       0.60      0.75      0.67        16
           1       0.64      0.47      0.54        15

    accuracy                           0.61        31
   macro avg       0.62      0.61      0.60        31
weighted avg       0.62      0.61      0.60        31

Confusion matrix (rows = true, cols = predicted):
[[12  4]
 [ 8  7]]
Fold 1 macro F1: 0.6026

=== Fold 2 / 5 ===
              precision    recall  f1-score   support

           0       0.67      0.88      0.76        16
           1       0.80      0.53      0.64        15

    accuracy                           0.71        31
   ma